In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
from datetime import datetime, date, time

# Example 1: Basic Database Data Types
print("Example 1: Basic Database Data Types")

try:
    print("✓ db_dtypes package information:")
    print(f"  Version: {db_dtypes.__version__ if hasattr(db_dtypes, '__version__') else 'Unknown'}")
    
    # Check available data types
    available_types = []
    for attr in dir(db_dtypes):
        if not attr.startswith('_') and hasattr(getattr(db_dtypes, attr), '__name__'):
            available_types.append(attr)
    
    print(f"  Available types: {len(available_types)}")
    print(f"  Key types: {available_types[:5] if available_types else 'None found'}")
    
    # db_dtypes adds only the types pandas lacks (DateDtype, TimeDtype, JSONDtype); BigQuery STRING / INT64 /
    # BOOL columns use pandas' own nullable dtypes, which is what the next three sections exercise.
    assert {'DateDtype', 'TimeDtype', 'JSONDtype'} <= set(available_types), available_types
    
    # Test basic string type (most fundamental)
    print("\n✓ Testing StringDtype:")
    
    # Create a pandas Series with pandas' nullable StringDtype
    string_data = ["hello", "world", "database", "types", None]
    
    try:
        string_series = pd.Series(string_data, dtype=pd.StringDtype())
        print(f"  StringDtype series: {string_series.dtype}")
        print(f"  Data: {list(string_series)}")
        print(f"  Null handling: {string_series.isna().sum()} nulls")
        
        # Test string operations
        upper_series = string_series.str.upper()
        print(f"  Upper case: {list(upper_series.dropna())}")
        assert string_series.isna().sum() == 1
        assert list(upper_series.dropna()) == ["HELLO", "WORLD", "DATABASE", "TYPES"]
        
    except Exception as e:
        print(f"  StringDtype error: {e}")
        raise
    
    # Test numeric types
    print("\n✓ Testing numeric types:")
    
    # Integer data
    int_data = [1, 2, 3, None, 5]
    
    try:
        # pandas' nullable Int64Dtype (BigQuery INT64)
        int_series = pd.Series(int_data, dtype=pd.Int64Dtype())
        print(f"  Int64Dtype: {int_series.dtype}")
        print(f"  Data: {list(int_series)}")
        print(f"  Sum (excluding nulls): {int_series.sum()}")
        assert int_series.sum() == 11 and int_series.isna().sum() == 1
        
    except Exception as e:
        print(f"  Int64Dtype error: {e}")
        raise
    
    # Boolean data
    bool_data = [True, False, True, None, False]
    
    try:
        # pandas' nullable BooleanDtype (BigQuery BOOL)
        bool_series = pd.Series(bool_data, dtype=pd.BooleanDtype())
        print(f"  BooleanDtype: {bool_series.dtype}")
        print(f"  Data: {list(bool_series)}")
        print(f"  True count: {bool_series.sum()}")
        assert bool_series.sum() == 2 and bool_series.isna().sum() == 1
        
    except Exception as e:
        print(f"  BooleanDtype error: {e}")
        raise
    
    # Test DataFrame with mixed types
    print("\n✓ Testing DataFrame with db_dtypes:")
    
    try:
        df_data = {
            'id': pd.Series([1, 2, 3, 4, None], dtype=pd.Int64Dtype()),
            'name': pd.Series(["Alice", "Bob", "Charlie", None, "Eve"], dtype=pd.StringDtype()),
            'active': pd.Series([True, False, True, True, None], dtype=pd.BooleanDtype()),
            # a BigQuery DATE column: db_dtypes' own extension type
            'joined': pd.Series([date(2024, 1, 1), date(2024, 2, 1), None, date(2024, 4, 1), date(2024, 5, 1)],
                                dtype=db_dtypes.DateDtype())
        }
        
        df = pd.DataFrame(df_data)
        print(f"  DataFrame shape: {df.shape}")
        print(f"  Data types:\n{df.dtypes}")
        print(f"  Null counts:\n{df.isna().sum()}")
        print(f"  Sample data:\n{df.head()}")
        assert df.shape == (5, 4)
        assert isinstance(df['joined'].dtype, db_dtypes.DateDtype)
        assert df.isna().sum().to_dict() == {'id': 1, 'name': 1, 'active': 1, 'joined': 1}
        
    except Exception as e:
        print(f"  DataFrame creation error: {e}")
        raise
    
    # Test type information
    print("\n✓ Type information:")
    
    for dtype_name in ['StringDtype', 'Int64Dtype', 'BooleanDtype']:
        if hasattr(db_dtypes, dtype_name):
            dtype_class = getattr(db_dtypes, dtype_name)
            print(f"  {dtype_name}: Available")
            
            # Create instance to test properties
            try:
                dtype_instance = dtype_class()
                print(f"    Name: {dtype_instance.name}")
                print(f"    Kind: {getattr(dtype_instance, 'kind', 'Unknown')}")
                print(f"    NA value: {getattr(dtype_instance, 'na_value', 'Unknown')}")
            except Exception as e:
                print(f"    Instance creation error: {e}")
                raise
        else:
            print(f"  {dtype_name}: Not available")
    
    print("✓ Basic database data types completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: db_dtypes package may not be installed")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"db_dtypes example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
from datetime import datetime, date, time
import decimal

# Example 2: BigQuery Data Types
print("Example 2: BigQuery Data Types")

try:
    print("✓ Testing BigQuery-specific data types:")
    
    # Test JSON type if available
    print("\n✓ Testing JSON type:")
    try:
        json_data = ['{"name": "Alice", "age": 30}', '{"name": "Bob", "age": 25}', None, '{"name": "Charlie"}']
        
        # Try to use JSON type from db_dtypes
        if hasattr(db_dtypes, 'JSONDtype'):
            json_series = pd.Series(json_data, dtype=db_dtypes.JSONDtype())
            print(f"  JSONDtype series: {json_series.dtype}")
            print(f"  Data: {list(json_series)}")
            print(f"  Non-null count: {json_series.count()}")
        else:
            print("  JSONDtype not available, using string fallback")
            json_series = pd.Series(json_data, dtype="string")
            print(f"  String fallback: {json_series.dtype}")
            
    except Exception as e:
        print(f"  JSON type error: {e}")
        raise
    
    # Test BYTES type
    print("\n✓ Testing BYTES type:")
    try:
        bytes_data = [b'hello', b'world', None, b'binary data', b'\x00\x01\x02']
        
        if hasattr(db_dtypes, 'BytesDtype'):
            bytes_series = pd.Series(bytes_data, dtype=db_dtypes.BytesDtype())
            print(f"  BytesDtype series: {bytes_series.dtype}")
            print(f"  Data length: {len(bytes_series)}")
            print(f"  Sample: {bytes_series.iloc[0] if not pd.isna(bytes_series.iloc[0]) else 'None'}")
        else:
            print("  BytesDtype not available")
            # Fallback: object dtype for bytes
            bytes_series = pd.Series(bytes_data, dtype=object)
            print(f"  Object fallback: {bytes_series.dtype}")
            
    except Exception as e:
        print(f"  BYTES type error: {e}")
        raise
    
    # Test DECIMAL/NUMERIC type
    print("\n✓ Testing DECIMAL/NUMERIC type:")
    try:
        decimal_data = [
            decimal.Decimal('123.456'),
            decimal.Decimal('789.012'),
            None,
            decimal.Decimal('0.001'),
            decimal.Decimal('999999.999999')
        ]
        
        if hasattr(db_dtypes, 'DecimalDtype'):
            decimal_series = pd.Series(decimal_data, dtype=db_dtypes.DecimalDtype())
            print(f"  DecimalDtype series: {decimal_series.dtype}")
            print(f"  Data: {list(decimal_series)}")
            print(f"  Sum: {decimal_series.sum()}")
        else:
            print("  DecimalDtype not available")
            # Fallback: object dtype
            decimal_series = pd.Series(decimal_data, dtype=object)
            print(f"  Object fallback: {decimal_series.dtype}")
            print(f"  Sample precision: {decimal_data[0]}")
            
    except Exception as e:
        print(f"  DECIMAL type error: {e}")
        raise
    
    # Test DATE type
    print("\n✓ Testing DATE type:")
    try:
        date_data = [
            date(2024, 1, 15),
            date(2024, 2, 29),  # Leap year
            None,
            date(2024, 12, 31),
            date(2023, 6, 15)
        ]
        
        if hasattr(db_dtypes, 'DateDtype'):
            date_series = pd.Series(date_data, dtype=db_dtypes.DateDtype())
            print(f"  DateDtype series: {date_series.dtype}")
            print(f"  Data: {list(date_series)}")
            print(f"  Date range: {date_series.min()} to {date_series.max()}")
        else:
            print("  DateDtype not available")
            # Fallback: object dtype
            date_series = pd.Series(date_data, dtype=object)
            print(f"  Object fallback: {date_series.dtype}")
            
    except Exception as e:
        print(f"  DATE type error: {e}")
        raise
    
    # Test TIME type
    print("\n✓ Testing TIME type:")
    try:
        time_data = [
            time(9, 30, 0),
            time(14, 45, 30),
            None,
            time(23, 59, 59),
            time(0, 0, 0)
        ]
        
        if hasattr(db_dtypes, 'TimeDtype'):
            time_series = pd.Series(time_data, dtype=db_dtypes.TimeDtype())
            print(f"  TimeDtype series: {time_series.dtype}")
            print(f"  Data: {list(time_series)}")
        else:
            print("  TimeDtype not available")
            # Fallback: object dtype
            time_series = pd.Series(time_data, dtype=object)
            print(f"  Object fallback: {time_series.dtype}")
            
    except Exception as e:
        print(f"  TIME type error: {e}")
        raise
    
    # Test DATETIME type
    print("\n✓ Testing DATETIME type:")
    try:
        datetime_data = [
            datetime(2024, 1, 15, 14, 30, 0),
            datetime(2024, 2, 29, 9, 15, 45),
            None,
            datetime(2024, 12, 31, 23, 59, 59),
            datetime(2023, 6, 15, 12, 0, 0)
        ]
        
        if hasattr(db_dtypes, 'DatetimeDtype'):
            datetime_series = pd.Series(datetime_data, dtype=db_dtypes.DatetimeDtype())
            print(f"  DatetimeDtype series: {datetime_series.dtype}")
            print(f"  Data count: {datetime_series.count()}")
            print(f"  Sample: {datetime_series.iloc[0]}")
        else:
            print("  DatetimeDtype not available")
            # Fallback: pandas datetime64
            datetime_series = pd.Series(datetime_data, dtype='datetime64[ns]')
            print(f"  Datetime64 fallback: {datetime_series.dtype}")
            
    except Exception as e:
        print(f"  DATETIME type error: {e}")
        raise
    
    # Test BigQuery DataFrame simulation
    print("\n✓ BigQuery-style DataFrame:")
    try:
        # Create a DataFrame that simulates BigQuery table structure
        bq_data = {
            'id': pd.Series([1, 2, 3, None, 5], dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
            'name': pd.Series(["Alice", "Bob", None, "Diana", "Eve"], dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
            'created_date': pd.Series([date(2024, 1, 1), date(2024, 1, 2), date(2024, 1, 3), None, date(2024, 1, 5)]),
            'is_active': pd.Series([True, False, True, None, True], dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean'),
            'metadata': pd.Series(['{"type": "user"}', '{"type": "admin"}', None, '{"type": "guest"}', '{"type": "user"}'])
        }
        
        bq_df = pd.DataFrame(bq_data)
        print(f"  BigQuery-style DataFrame shape: {bq_df.shape}")
        print(f"  Column types:\n{bq_df.dtypes}")
        print(f"  Memory usage:\n{bq_df.memory_usage(deep=True)}")
        print(f"  Info summary:")
        bq_df.info(memory_usage='deep')
        
        # Test some BigQuery-like operations
        print(f"\n  Query-like operations:")
        active_users = bq_df[bq_df['is_active'] == True]
        print(f"    Active users: {len(active_users)}")
        
        non_null_names = bq_df[bq_df['name'].notna()]
        print(f"    Non-null names: {len(non_null_names)}")
        
    except Exception as e:
        print(f"  BigQuery DataFrame error: {e}")
        raise
    
    # Test type compatibility
    print("\n✓ Type compatibility testing:")
    
    type_tests = [
        ('StringDtype', ["test", None, "value"]),
        ('Int64Dtype', [1, None, 3]),
        ('BooleanDtype', [True, None, False]),
        ('Float64Dtype', [1.5, None, 3.7])
    ]
    
    for dtype_name, test_data in type_tests:
        if hasattr(db_dtypes, dtype_name):
            try:
                dtype_class = getattr(db_dtypes, dtype_name)
                test_series = pd.Series(test_data, dtype=dtype_class())
                print(f"  {dtype_name}: ✓ Compatible")
                print(f"    Null handling: {test_series.isna().sum()} nulls")
                print(f"    Memory efficient: {test_series.memory_usage(deep=True)} bytes")
            except Exception as e:
                print(f"  {dtype_name}: ✗ Error - {e}")
                raise
        else:
            print(f"  {dtype_name}: Not available")
    
    print("✓ BigQuery data types completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"db_dtypes example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
import io

# Example 3: Pandas Integration and Operations
print("Example 3: Pandas Integration and Operations")

try:
    print("✓ Advanced pandas integration with db_dtypes:")
    
    # Create a comprehensive dataset using db_dtypes
    print("\n✓ Creating dataset with multiple db_dtypes:")
    
    # Sample e-commerce data
    ecommerce_data = {
        'order_id': pd.Series([1001, 1002, 1003, None, 1005, 1006], 
                             dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
        'customer_name': pd.Series(["Alice Johnson", "Bob Smith", None, "Diana Prince", "Eve Adams", "Frank Miller"], 
                                  dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
        'product_price': pd.Series([29.99, 15.50, 99.99, None, 45.00, 12.99], 
                                  dtype=db_dtypes.Float64Dtype() if hasattr(db_dtypes, 'Float64Dtype') else 'Float64'),
        'is_premium': pd.Series([True, False, True, None, False, True], 
                               dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean'),
        'order_notes': pd.Series(["Express delivery", "", None, "Gift wrap", "Standard", ""], 
                                dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string')
    }
    
    df = pd.DataFrame(ecommerce_data)
    print(f"  Dataset shape: {df.shape}")
    print(f"  Data types:\n{df.dtypes}")
    
    # Test pandas operations with db_dtypes
    print("\n✓ Testing pandas operations:")
    
    # Filtering operations
    premium_orders = df[df['is_premium'] == True]
    print(f"  Premium orders: {len(premium_orders)}")
    
    # String operations
    if df['customer_name'].dtype.name in ['string', 'StringDtype']:
        name_lengths = df['customer_name'].str.len()
        print(f"  Average name length: {name_lengths.mean():.1f} characters")
        
        # String contains operation
        has_smith = df['customer_name'].str.contains('Smith', na=False)
        print(f"  Orders with 'Smith': {has_smith.sum()}")
    
    # Numeric operations
    if df['product_price'].dtype.name in ['Float64', 'Float64Dtype']:
        total_revenue = df['product_price'].sum()
        avg_price = df['product_price'].mean()
        print(f"  Total revenue: ${total_revenue:.2f}")
        print(f"  Average price: ${avg_price:.2f}")
        
        # Price categories
        price_categories = pd.cut(df['product_price'], bins=[0, 20, 50, 100], 
                                 labels=['Low', 'Medium', 'High'])
        print(f"  Price distribution:\n{price_categories.value_counts()}")
    
    # Null handling
    print("\n✓ Testing null handling:")
    
    null_counts = df.isnull().sum()
    print(f"  Null counts by column:\n{null_counts}")
    
    # Fill null values with appropriate defaults
    df_filled = df.copy()
    
    # Fill numeric nulls with median
    if 'product_price' in df_filled.columns:
        median_price = df_filled['product_price'].median()
        df_filled['product_price'] = df_filled['product_price'].fillna(median_price)
        print(f"  Filled price nulls with median: ${median_price:.2f}")
    
    # Fill string nulls with default
    if 'customer_name' in df_filled.columns:
        df_filled['customer_name'] = df_filled['customer_name'].fillna("Unknown Customer")
        print(f"  Filled name nulls with default value")
    
    # Fill boolean nulls with False
    if 'is_premium' in df_filled.columns:
        df_filled['is_premium'] = df_filled['is_premium'].fillna(False)
        print(f"  Filled boolean nulls with False")
    
    print(f"  Nulls after filling:\n{df_filled.isnull().sum()}")
    
    # Grouping operations
    print("\n✓ Testing grouping operations:")
    
    try:
        # Group by premium status
        premium_groups = df.groupby('is_premium', dropna=False)
        
        for name, group in premium_groups:
            print(f"  Premium={name}: {len(group)} orders")
            if len(group) > 0 and 'product_price' in group.columns:
                avg_price_group = group['product_price'].mean()
                if not pd.isna(avg_price_group):
                    print(f"    Average price: ${avg_price_group:.2f}")
                
    except Exception as e:
        print(f"  Grouping error: {e}")
        raise
    
    # Merge operations
    print("\n✓ Testing merge operations:")
    
    try:
        # Create a customer info DataFrame
        customer_info = pd.DataFrame({
            'customer_name': pd.Series(["Alice Johnson", "Bob Smith", "Diana Prince", "New Customer"], 
                                      dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
            'customer_tier': pd.Series(["Gold", "Silver", "Platinum", "Bronze"], 
                                      dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
            'loyalty_points': pd.Series([1500, 800, 2500, 100], 
                                       dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64')
        })
        
        # Merge with main DataFrame
        merged_df = df.merge(customer_info, on='customer_name', how='left')
        print(f"  Merged dataset shape: {merged_df.shape}")
        print(f"  New columns: {list(merged_df.columns)}")
        
        # Show merge results
        tier_counts = merged_df['customer_tier'].value_counts(dropna=False)
        print(f"  Customer tier distribution:\n{tier_counts}")
        
    except Exception as e:
        print(f"  Merge error: {e}")
        raise
    
    # Pivot operations
    print("\n✓ Testing pivot operations:")
    
    try:
        # Create summary pivot
        if 'is_premium' in df.columns and 'product_price' in df.columns:
            pivot_summary = df.pivot_table(
                values='product_price', 
                index='is_premium', 
                aggfunc=['count', 'mean', 'sum'],
                dropna=False
            )
            print(f"  Pivot table shape: {pivot_summary.shape}")
            print(f"  Pivot summary:\n{pivot_summary}")
            
    except Exception as e:
        print(f"  Pivot error: {e}")
        raise
    
    # I/O operations
    print("\n✓ Testing I/O operations:")
    
    try:
        # Test CSV export/import with db_dtypes
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False)
        csv_content = csv_buffer.getvalue()
        print(f"  CSV export successful: {len(csv_content)} characters")
        
        # Test reading back
        csv_buffer.seek(0)
        df_from_csv = pd.read_csv(csv_buffer)
        print(f"  CSV import shape: {df_from_csv.shape}")
        print(f"  CSV import dtypes:\n{df_from_csv.dtypes}")
        
        # Convert back to db_dtypes
        if hasattr(db_dtypes, 'StringDtype'):
            df_from_csv['customer_name'] = df_from_csv['customer_name'].astype(db_dtypes.StringDtype())
            df_from_csv['order_notes'] = df_from_csv['order_notes'].astype(db_dtypes.StringDtype())
        
        if hasattr(db_dtypes, 'Int64Dtype'):
            df_from_csv['order_id'] = df_from_csv['order_id'].astype(db_dtypes.Int64Dtype())
        
        print(f"  Converted dtypes:\n{df_from_csv.dtypes}")
        
    except Exception as e:
        print(f"  I/O error: {e}")
        raise
    
    # Performance comparison
    print("\n✓ Performance comparison:")
    
    try:
        import time
        
        # Large dataset test
        large_size = 10000
        
        # Create large dataset with regular pandas types
        start_time = time.time()
        large_data_regular = pd.DataFrame({
            'id': range(large_size),
            'name': [f"User_{i}" for i in range(large_size)],
            'value': np.random.randn(large_size),
            'flag': np.random.choice([True, False], large_size)
        })
        regular_time = time.time() - start_time
        regular_memory = large_data_regular.memory_usage(deep=True).sum()
        
        # Create large dataset with db_dtypes
        start_time = time.time()
        large_data_db = pd.DataFrame({
            'id': pd.Series(range(large_size), dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
            'name': pd.Series([f"User_{i}" for i in range(large_size)], dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
            'value': pd.Series(np.random.randn(large_size), dtype=db_dtypes.Float64Dtype() if hasattr(db_dtypes, 'Float64Dtype') else 'Float64'),
            'flag': pd.Series(np.random.choice([True, False], large_size), dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean')
        })
        db_time = time.time() - start_time
        db_memory = large_data_db.memory_usage(deep=True).sum()
        
        print(f"  Large dataset ({large_size:,} rows):")
        print(f"    Regular pandas: {regular_time:.3f}s, {regular_memory:,} bytes")
        print(f"    db_dtypes: {db_time:.3f}s, {db_memory:,} bytes")
        print(f"    Memory ratio: {db_memory/regular_memory:.2f}x")
        
    except Exception as e:
        print(f"  Performance test error: {e}")
        raise
    
    print("✓ Pandas integration and operations completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"db_dtypes example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
from datetime import datetime, date

# Example 4: Type Conversion and Validation
print("Example 4: Type Conversion and Validation")

try:
    print("✓ Testing type conversion and validation:")
    
    # Raw data with mixed types that need conversion
    print("\n✓ Converting mixed data types:")
    
    raw_data = {
        'user_id': ['1', '2', '3', '', '5', 'invalid'],  # String numbers with empty and invalid
        'username': [123, 'alice', None, 'bob', '', 456],  # Mixed types for strings
        'age': ['25', '30.5', 'invalid', '', None, '45'],  # String numbers with issues
        'is_active': ['true', 'false', 1, 0, '', None],   # Mixed boolean representations
        'score': [85.5, '90', 'high', None, '', 75.2]    # Mixed numeric types
    }
    
    raw_df = pd.DataFrame(raw_data)
    print(f"  Raw data shape: {raw_df.shape}")
    print(f"  Raw dtypes:\n{raw_df.dtypes}")
    print(f"  Sample raw data:\n{raw_df.head()}")
    
    # Safe conversion functions
    def safe_convert_to_int64(series):
        """Safely convert series to Int64Dtype with db_dtypes"""
        try:
            # First convert to numeric, coercing errors to NaN
            numeric_series = pd.to_numeric(series, errors='coerce')
            
            # Convert to Int64 (nullable integer)
            if hasattr(db_dtypes, 'Int64Dtype'):
                return numeric_series.astype(db_dtypes.Int64Dtype())
            else:
                return numeric_series.astype('Int64')
        except Exception as e:
            print(f"    Int64 conversion error: {e}")
            return series
    
    def safe_convert_to_string(series):
        """Safely convert series to StringDtype with db_dtypes"""
        try:
            # Convert to string, handling None values
            str_series = series.astype(str)
            str_series = str_series.replace('nan', pd.NA)
            str_series = str_series.replace('None', pd.NA)
            str_series = str_series.replace('', pd.NA)
            
            if hasattr(db_dtypes, 'StringDtype'):
                return str_series.astype(db_dtypes.StringDtype())
            else:
                return str_series.astype('string')
        except Exception as e:
            print(f"    String conversion error: {e}")
            return series
    
    def safe_convert_to_float64(series):
        """Safely convert series to Float64Dtype with db_dtypes"""
        try:
            # Convert to numeric, coercing errors to NaN
            numeric_series = pd.to_numeric(series, errors='coerce')
            
            if hasattr(db_dtypes, 'Float64Dtype'):
                return numeric_series.astype(db_dtypes.Float64Dtype())
            else:
                return numeric_series.astype('Float64')
        except Exception as e:
            print(f"    Float64 conversion error: {e}")
            return series
    
    def safe_convert_to_boolean(series):
        """Safely convert series to BooleanDtype with db_dtypes"""
        try:
            # Handle various boolean representations
            bool_map = {
                'true': True, 'false': False,
                'True': True, 'False': False,
                '1': True, '0': False,
                1: True, 0: False,
                'yes': True, 'no': False,
                'y': True, 'n': False
            }
            
            mapped_series = series.map(bool_map)
            
            if hasattr(db_dtypes, 'BooleanDtype'):
                return mapped_series.astype(db_dtypes.BooleanDtype())
            else:
                return mapped_series.astype('boolean')
        except Exception as e:
            print(f"    Boolean conversion error: {e}")
            return series
    
    # Apply conversions
    print("\n✓ Applying safe conversions:")
    
    converted_df = raw_df.copy()
    
    # Convert user_id to Int64
    print("  Converting user_id to Int64...")
    converted_df['user_id'] = safe_convert_to_int64(raw_df['user_id'])
    
    # Convert username to String
    print("  Converting username to String...")
    converted_df['username'] = safe_convert_to_string(raw_df['username'])
    
    # Convert age to Float64 (age could have decimals)
    print("  Converting age to Float64...")
    converted_df['age'] = safe_convert_to_float64(raw_df['age'])
    
    # Convert is_active to Boolean
    print("  Converting is_active to Boolean...")
    converted_df['is_active'] = safe_convert_to_boolean(raw_df['is_active'])
    
    # Convert score to Float64
    print("  Converting score to Float64...")
    converted_df['score'] = safe_convert_to_float64(raw_df['score'])
    
    print(f"\n  Converted dtypes:\n{converted_df.dtypes}")
    print(f"  Converted data:\n{converted_df}")
    
    # Validation results
    print("\n✓ Validation results:")
    
    for column in converted_df.columns:
        null_count = converted_df[column].isna().sum()
        total_count = len(converted_df[column])
        success_rate = ((total_count - null_count) / total_count) * 100
        print(f"  {column}: {success_rate:.1f}% successful conversion ({null_count} nulls)")
    
    # Advanced validation with custom rules
    print("\n✓ Custom validation rules:")
    
    def validate_user_data(df):
        """Apply custom validation rules"""
        validation_results = {}
        
        # Rule 1: user_id should be positive
        if 'user_id' in df.columns:
            valid_ids = df['user_id'] > 0
            validation_results['positive_user_id'] = valid_ids.sum()
            print(f"    Positive user IDs: {valid_ids.sum()}/{len(df)}")
        
        # Rule 2: username should not be empty after conversion
        if 'username' in df.columns:
            valid_usernames = df['username'].notna() & (df['username'] != '')
            validation_results['valid_usernames'] = valid_usernames.sum()
            print(f"    Valid usernames: {valid_usernames.sum()}/{len(df)}")
        
        # Rule 3: age should be reasonable (0-150)
        if 'age' in df.columns:
            reasonable_ages = (df['age'] >= 0) & (df['age'] <= 150)
            validation_results['reasonable_ages'] = reasonable_ages.sum()
            print(f"    Reasonable ages: {reasonable_ages.sum()}/{len(df)}")
        
        # Rule 4: score should be 0-100
        if 'score' in df.columns:
            valid_scores = (df['score'] >= 0) & (df['score'] <= 100)
            validation_results['valid_scores'] = valid_scores.sum()
            print(f"    Valid scores (0-100): {valid_scores.sum()}/{len(df)}")
        
        return validation_results
    
    validation_results = validate_user_data(converted_df)
    
    # Type-specific operations to verify conversions worked
    print("\n✓ Testing type-specific operations:")
    
    try:
        # String operations
        if 'username' in converted_df.columns:
            username_lengths = converted_df['username'].str.len()
            print(f"    Username lengths: {list(username_lengths.dropna())}")
            
            uppercase_usernames = converted_df['username'].str.upper()
            print(f"    Uppercase usernames: {list(uppercase_usernames.dropna())}")
        
        # Numeric operations
        if 'age' in converted_df.columns:
            avg_age = converted_df['age'].mean()
            print(f"    Average age: {avg_age:.1f}")
            
            age_categories = pd.cut(converted_df['age'], bins=[0, 18, 35, 65, 150], 
                                   labels=['Child', 'Young Adult', 'Adult', 'Senior'])
            print(f"    Age categories: {age_categories.value_counts().to_dict()}")
        
        # Boolean operations
        if 'is_active' in converted_df.columns:
            active_count = converted_df['is_active'].sum()
            total_non_null = converted_df['is_active'].count()
            print(f"    Active users: {active_count}/{total_non_null}")
            
            # Boolean mask operation
            active_users = converted_df[converted_df['is_active'] == True]
            print(f"    Active user data shape: {active_users.shape}")
        
    except Exception as e:
        print(f"    Type operations error: {e}")
        raise
    
    # Error handling and recovery
    print("\n✓ Error handling scenarios:")
    
    # Test with completely invalid data
    invalid_data = {
        'numbers': ['abc', 'def', 'ghi'],
        'booleans': ['maybe', 'perhaps', 'unknown'],
        'dates': ['not-a-date', '32/13/2024', 'yesterday']
    }
    
    invalid_df = pd.DataFrame(invalid_data)
    print(f"  Invalid data input:\n{invalid_df}")
    
    # Try conversions with error handling
    try:
        invalid_numbers = safe_convert_to_int64(invalid_df['numbers'])
        print(f"    Invalid numbers result: {list(invalid_numbers)} (all NaN expected)")
        
        invalid_booleans = safe_convert_to_boolean(invalid_df['booleans'])
        print(f"    Invalid booleans result: {list(invalid_booleans)} (all NaN expected)")
        
    except Exception as e:
        print(f"    Error handling test error: {e}")
        raise
    
    # Best practices summary
    print("\n✓ Type conversion best practices:")
    
    best_practices = [
        "Always use pd.to_numeric with errors='coerce' for numeric conversions",
        "Handle empty strings separately from None/NaN values",
        "Use mapping for boolean conversions to handle various representations",
        "Validate data ranges after conversion (age 0-150, scores 0-100, etc.)",
        "Prefer nullable dtypes (Int64, Float64, boolean, string) over numpy types",
        "Test conversion functions with edge cases and invalid data",
        "Use db_dtypes for better database compatibility and null handling"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"    {i}. {practice}")
    
    print("✓ Type conversion and validation completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"db_dtypes example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
from datetime import datetime, date, time
import sqlite3
import io

# Example 5: Real-world Database Operations
print("Example 5: Real-world Database Operations")

try:
    print("✓ Simulating real-world database operations with db_dtypes:")
    
    # Create sample data that mimics database tables
    print("\n✓ Creating sample database-like tables:")
    
    # Users table
    users_data = {
        'user_id': pd.Series([1, 2, 3, 4, 5], dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
        'username': pd.Series(["alice_smith", "bob_jones", "charlie_brown", "diana_prince", "eve_adams"], 
                             dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
        'email': pd.Series(["alice@example.com", "bob@example.com", "charlie@example.com", "diana@example.com", "eve@example.com"], 
                          dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
        'is_premium': pd.Series([True, False, True, False, True], 
                               dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean'),
        'created_at': pd.Series([
            datetime(2024, 1, 15, 10, 30, 0),
            datetime(2024, 1, 16, 14, 45, 0),
            datetime(2024, 1, 17, 9, 15, 0),
            datetime(2024, 1, 18, 16, 20, 0),
            datetime(2024, 1, 19, 11, 0, 0)
        ]),
        'last_login': pd.Series([
            datetime(2024, 1, 20, 8, 30, 0),
            None,
            datetime(2024, 1, 21, 19, 45, 0),
            datetime(2024, 1, 20, 12, 15, 0),
            None
        ])
    }
    
    users_df = pd.DataFrame(users_data)
    print(f"  Users table: {users_df.shape}")
    print(f"  Users dtypes:\n{users_df.dtypes}")
    
    # Orders table
    orders_data = {
        'order_id': pd.Series([101, 102, 103, 104, 105, 106], 
                             dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
        'user_id': pd.Series([1, 2, 1, 3, 4, 1], 
                            dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
        'product_name': pd.Series(["Laptop", "Mouse", "Keyboard", "Monitor", "Tablet", "Headphones"], 
                                 dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
        'amount': pd.Series([999.99, 25.50, 79.99, 299.99, 599.00, 149.95], 
                           dtype=db_dtypes.Float64Dtype() if hasattr(db_dtypes, 'Float64Dtype') else 'Float64'),
        'quantity': pd.Series([1, 2, 1, 1, 1, 1], 
                             dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
        'order_date': pd.Series([
            datetime(2024, 1, 20, 10, 0, 0),
            datetime(2024, 1, 20, 14, 30, 0),
            datetime(2024, 1, 21, 9, 15, 0),
            datetime(2024, 1, 21, 11, 45, 0),
            datetime(2024, 1, 22, 15, 20, 0),
            datetime(2024, 1, 22, 16, 10, 0)
        ]),
        'is_shipped': pd.Series([True, True, False, True, False, False], 
                               dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean')
    }
    
    orders_df = pd.DataFrame(orders_data)
    print(f"  Orders table: {orders_df.shape}")
    print(f"  Orders dtypes:\n{orders_df.dtypes}")
    
    # Simulate database queries with pandas operations
    print("\n✓ Simulating SQL-like operations:")
    
    # Query 1: SELECT * FROM users WHERE is_premium = True
    premium_users = users_df[users_df['is_premium'] == True]
    print(f"  Premium users query result: {len(premium_users)} users")
    print(f"    Premium usernames: {list(premium_users['username'])}")
    
    # Query 2: JOIN users and orders
    user_orders = orders_df.merge(users_df, on='user_id', how='left')
    print(f"  User-Orders JOIN result: {user_orders.shape}")
    print(f"    Sample joined data:")
    print(f"    {user_orders[['username', 'product_name', 'amount']].head()}")
    
    # Query 3: GROUP BY with aggregations
    user_summary = user_orders.groupby('username').agg({
        'amount': ['sum', 'count', 'mean'],
        'quantity': 'sum',
        'is_shipped': lambda x: (x == True).sum()
    }).round(2)
    
    # Flatten column names
    user_summary.columns = ['_'.join(col).strip() for col in user_summary.columns]
    print(f"  User summary (GROUP BY):\n{user_summary}")
    
    # Query 4: Date-based filtering
    recent_orders = orders_df[orders_df['order_date'] >= datetime(2024, 1, 21)]
    print(f"  Recent orders (>= 2024-01-21): {len(recent_orders)} orders")
    
    # Data export simulation
    print("\n✓ Simulating data export operations:")
    
    try:
        # Export to CSV with proper handling of db_dtypes
        csv_buffer = io.StringIO()
        users_df.to_csv(csv_buffer, index=False)
        csv_content = csv_buffer.getvalue()
        print(f"  Users CSV export: {len(csv_content)} characters")
        
        # Simulate reading back from CSV
        csv_buffer.seek(0)
        users_from_csv = pd.read_csv(csv_buffer)
        
        # Re-apply db_dtypes after CSV import
        type_conversions = {
            'user_id': db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64',
            'username': db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string',
            'email': db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string',
            'is_premium': db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean'
        }
        
        for col, dtype in type_conversions.items():
            if col in users_from_csv.columns:
                users_from_csv[col] = users_from_csv[col].astype(dtype)
        
        print(f"  CSV import with type conversion successful")
        print(f"  Restored dtypes:\n{users_from_csv.dtypes}")
        
    except Exception as e:
        print(f"  CSV export/import error: {e}")
        raise
    
    # Database schema simulation
    print("\n✓ Database schema simulation:")
    
    def get_db_schema(df, table_name):
        """Generate database-like schema information"""
        schema_info = {
            'table_name': table_name,
            'column_count': len(df.columns),
            'row_count': len(df),
            'columns': []
        }
        
        for col in df.columns:
            col_info = {
                'name': col,
                'dtype': str(df[col].dtype),
                'nullable': df[col].isna().any(),
                'null_count': df[col].isna().sum(),
                'unique_count': df[col].nunique()
            }
            
            # Add type-specific information
            if df[col].dtype.name in ['Int64', 'Int64Dtype']:
                col_info['db_type'] = 'BIGINT'
                if not df[col].isna().all():
                    col_info['min_value'] = df[col].min()
                    col_info['max_value'] = df[col].max()
            elif df[col].dtype.name in ['Float64', 'Float64Dtype']:
                col_info['db_type'] = 'DOUBLE'
                if not df[col].isna().all():
                    col_info['min_value'] = df[col].min()
                    col_info['max_value'] = df[col].max()
            elif df[col].dtype.name in ['string', 'StringDtype']:
                col_info['db_type'] = 'VARCHAR'
                if not df[col].isna().all():
                    col_info['max_length'] = df[col].str.len().max()
                    col_info['avg_length'] = df[col].str.len().mean()
            elif df[col].dtype.name in ['boolean', 'BooleanDtype']:
                col_info['db_type'] = 'BOOLEAN'
                if not df[col].isna().all():
                    col_info['true_count'] = (df[col] == True).sum()
                    col_info['false_count'] = (df[col] == False).sum()
            elif 'datetime' in df[col].dtype.name:
                col_info['db_type'] = 'TIMESTAMP'
                if not df[col].isna().all():
                    col_info['min_date'] = df[col].min()
                    col_info['max_date'] = df[col].max()
            else:
                col_info['db_type'] = 'TEXT'
            
            schema_info['columns'].append(col_info)
        
        return schema_info
    
    # Generate schema for both tables
    users_schema = get_db_schema(users_df, 'users')
    orders_schema = get_db_schema(orders_df, 'orders')
    
    print(f"  Users table schema:")
    print(f"    Rows: {users_schema['row_count']}, Columns: {users_schema['column_count']}")
    for col in users_schema['columns'][:3]:  # Show first 3 columns
        print(f"    {col['name']}: {col['db_type']} (nullable: {col['nullable']})")
    
    print(f"  Orders table schema:")
    print(f"    Rows: {orders_schema['row_count']}, Columns: {orders_schema['column_count']}")
    for col in orders_schema['columns'][:3]:  # Show first 3 columns
        print(f"    {col['name']}: {col['db_type']} (nullable: {col['nullable']})")
    
    # Data quality checks
    print("\n✓ Data quality checks:")
    
    def run_data_quality_checks(df, table_name):
        """Run comprehensive data quality checks"""
        print(f"  Quality checks for {table_name}:")
        
        # Check for duplicates
        duplicate_count = df.duplicated().sum()
        print(f"    Duplicate rows: {duplicate_count}")
        
        # Check for null values
        null_columns = df.columns[df.isna().any()].tolist()
        print(f"    Columns with nulls: {null_columns}")
        
        # Check for empty strings in string columns
        for col in df.columns:
            if df[col].dtype.name in ['string', 'StringDtype']:
                empty_strings = (df[col] == '').sum()
                if empty_strings > 0:
                    print(f"    Empty strings in {col}: {empty_strings}")
        
        # Check for negative values in numeric columns
        for col in df.columns:
            if df[col].dtype.name in ['Int64', 'Float64', 'Int64Dtype', 'Float64Dtype']:
                negative_values = (df[col] < 0).sum()
                if negative_values > 0:
                    print(f"    Negative values in {col}: {negative_values}")
        
        # Check data consistency
        if table_name == 'orders' and 'amount' in df.columns and 'quantity' in df.columns:
            zero_amount_orders = (df['amount'] == 0).sum()
            zero_quantity_orders = (df['quantity'] == 0).sum()
            print(f"    Zero amount orders: {zero_amount_orders}")
            print(f"    Zero quantity orders: {zero_quantity_orders}")
    
    run_data_quality_checks(users_df, 'users')
    run_data_quality_checks(orders_df, 'orders')
    
    # Performance analysis
    print("\n✓ Performance analysis:")
    
    try:
        import time
        
        # Benchmark operations with db_dtypes vs regular pandas
        test_size = 1000
        
        # Create test data with regular pandas types
        start_time = time.time()
        regular_data = pd.DataFrame({
            'id': range(test_size),
            'name': [f"user_{i}" for i in range(test_size)],
            'value': np.random.randn(test_size),
            'flag': np.random.choice([True, False], test_size)
        })
        regular_ops_time = time.time()
        
        # Perform operations
        regular_filtered = regular_data[regular_data['flag'] == True]
        regular_grouped = regular_data.groupby('flag')['value'].mean()
        regular_total_time = time.time() - start_time
        
        # Create test data with db_dtypes
        start_time = time.time()
        db_data = pd.DataFrame({
            'id': pd.Series(range(test_size), dtype=db_dtypes.Int64Dtype() if hasattr(db_dtypes, 'Int64Dtype') else 'Int64'),
            'name': pd.Series([f"user_{i}" for i in range(test_size)], dtype=db_dtypes.StringDtype() if hasattr(db_dtypes, 'StringDtype') else 'string'),
            'value': pd.Series(np.random.randn(test_size), dtype=db_dtypes.Float64Dtype() if hasattr(db_dtypes, 'Float64Dtype') else 'Float64'),
            'flag': pd.Series(np.random.choice([True, False], test_size), dtype=db_dtypes.BooleanDtype() if hasattr(db_dtypes, 'BooleanDtype') else 'boolean')
        })
        db_ops_time = time.time()
        
        # Perform operations
        db_filtered = db_data[db_data['flag'] == True]
        db_grouped = db_data.groupby('flag')['value'].mean()
        db_total_time = time.time() - start_time
        
        print(f"  Performance comparison ({test_size:,} rows):")
        print(f"    Regular pandas: {regular_total_time:.3f}s")
        print(f"    db_dtypes: {db_total_time:.3f}s")
        print(f"    Ratio: {db_total_time/regular_total_time:.2f}x")
        
        # Memory usage comparison
        regular_memory = regular_data.memory_usage(deep=True).sum()
        db_memory = db_data.memory_usage(deep=True).sum()
        print(f"    Memory usage - Regular: {regular_memory:,} bytes")
        print(f"    Memory usage - db_dtypes: {db_memory:,} bytes")
        print(f"    Memory ratio: {db_memory/regular_memory:.2f}x")
        
    except Exception as e:
        print(f"  Performance analysis error: {e}")
        raise
    
    print("✓ Real-world database operations completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"db_dtypes example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")